In [1]:
from pathlib import Path
import xarray as xr
import rioxarray  # Required for GeoTIFF support
import numpy as np

In [4]:
def count_problematic_values(directory, threshold=1e10):
    """
    Count problematic values in GeoTIFF files.
    Only prints output for files with extreme values.
    
    Parameters:
    -----------
    directory : str or Path
        Directory containing GeoTIFF files
    threshold : float
        Values with absolute value > threshold are considered problematic
    """
    directory = Path(directory)
    results = []
    
    for tif_file in directory.glob("*.tif"):
        try:
            # Open the GeoTIFF with xarray
            da = xr.open_dataarray(tif_file, engine='rasterio')
            
            # Count problematic values per band
            band_stats = {}
            for band_idx in range(da.shape[0]):
                band_data = da[band_idx].values
                
                # Count extreme values
                extreme_mask = np.abs(band_data) > threshold
                n_extreme = np.sum(extreme_mask)
                
                # Get value range
                valid_data = band_data[~extreme_mask]
                if valid_data.size > 0:
                    data_min = np.min(valid_data)
                    data_max = np.max(valid_data)
                else:
                    data_min = data_max = np.nan
                
                band_stats[f'band_{band_idx}'] = {
                    'n_extreme': n_extreme,
                    'n_total': band_data.size,
                    'pct_extreme': (n_extreme / band_data.size) * 100,
                    'valid_min': data_min,
                    'valid_max': data_max
                }
            
            # Overall statistics
            all_data = da.values.flatten()
            total_extreme = np.sum(np.abs(all_data) > threshold)
            
            result = {
                'file': tif_file.name,
                'total_extreme_values': total_extreme,
                'total_pixels': all_data.size,
                'pct_extreme': (total_extreme / all_data.size) * 100,
                'global_min': np.min(all_data),
                'global_max': np.max(all_data),
                'band_stats': band_stats
            }
            results.append(result)
            
            # Only print if there are problematic values
            if total_extreme > 0:
                print(f"\n{tif_file.name}")
                print(f"  Total extreme values: {total_extreme:,} / {all_data.size:,} ({result['pct_extreme']:.2f}%)")
                print(f"  Global range: [{result['global_min']:.2e}, {result['global_max']:.2e}]")
                
                # Print per-band stats
                for band_name, stats in band_stats.items():
                    if stats['n_extreme'] > 0:
                        print(f"  {band_name}: {stats['n_extreme']:,} extreme values ({stats['pct_extreme']:.2f}%)")
            
            da.close()
            
        except Exception as e:
            print(f"Error processing {tif_file.name}: {e}")
    
    return results

In [5]:
# Usage
directory = "/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/kaguya_static_all_wac/sem_seg/chips/"
results = count_problematic_values(directory, threshold=1e10)

# Print summary at the end
n_problematic = sum(1 for r in results if r['total_extreme_values'] > 0)
print(f"\n{'='*60}")
print(f"Summary: {n_problematic} / {len(results)} files have problematic values")
print(f"{'='*60}")

# Optional: Export results to CSV
import pandas as pd
summary = pd.DataFrame([{
    'file': r['file'],
    'total_extreme': r['total_extreme_values'],
    'pct_extreme': r['pct_extreme'],
    'global_min': r['global_min'],
    'global_max': r['global_max']
} for r in results])
summary.to_csv('problematic_values_summary.csv', index=False)


M1485838007CE_r300_c150_input_wac_static_chip.tif
  Total extreme values: 5 / 1,080,000 (0.00%)
  Global range: [-4.23e+35, 1.97e+37]
  band_4: 5 extreme values (0.01%)

Summary: 1 / 626 files have problematic values
